                    USER
                      │
                      ▼
             🤖 Planner Agent
          (Creates a plan)
                      │
                      ▼
             🤖 Writer Agent
         (Writes Python code)
                      │
                      ▼
            🤖 Reviewer Agent
      (Reviews & requests changes)
                      │
          APPROVED? ──┐
             No       │ Yes
              │       ▼
              └──► 🤖 Executor Agent
                     (Runs the code)
                          │
                          ▼
                     Final Output



                     =======

                     Architecture

Import Libraries

In [1]:
from google.colab import userdata
from google import genai

Create Gemini Client

In [3]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY_2")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini Client Ready")

Gemini Client Ready


Planner Agent

In [4]:
def planner_agent(task):

    prompt = f"""
You are a Planner Agent.

Your job:

1. Understand the task.
2. Break it into clear steps.
3. Return ONLY the numbered plan.

Task:
{task}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

Writer Agent

In [5]:
def writer_agent(task, plan, feedback=""):

    prompt = f"""
You are a Python Developer.

Use the following plan to write Python code.

Task:
{task}

Plan:
{plan}

Reviewer Feedback:
{feedback}

Return ONLY executable Python code.
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

Reviewer Agent

In [6]:
def reviewer_agent(code):

    prompt = f"""
You are a Senior Python Reviewer.

Review the following code.

If it is correct, reply ONLY:

APPROVED

Otherwise explain what should be fixed.

Python Code:

{code}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

Executor Agent

In [7]:
def executor_agent(code):

    # Remove Markdown if present
    code = code.replace("```python", "")
    code = code.replace("```", "")
    code = code.strip()

    print("=" * 60)
    print("EXECUTOR AGENT")
    print("=" * 60)

    exec(code)

Executor Agent

In [8]:
def executor_agent(code):

    # Remove Markdown if present
    code = code.replace("```python", "")
    code = code.replace("```", "")
    code = code.strip()

    print("=" * 60)
    print("EXECUTOR AGENT")
    print("=" * 60)

    exec(code)

User Task

In [9]:
task = """
Student Marks

Tamil = 90
English = 85
Maths = 95
Science = 92
Social = 88

Calculate:

1. Total
2. Average
3. Percentage

Print the results.
"""

Planner

In [10]:
print("=" * 60)
print("PLANNER AGENT")
print("=" * 60)

plan = planner_agent(task)

print(plan)

PLANNER AGENT
1. Store the given marks for Tamil (90), English (85), Maths (95), Science (92), and Social (88).
2. Calculate the Total by adding all five subject marks together.
3. Calculate the Average by dividing the Total by the number of subjects (5).
4. Calculate the Percentage by dividing the Total by the maximum total marks (500) and multiplying by 100.
5. Display the results for Total, Average, and Percentage.


Writer & Reviewer Conversation

In [11]:
feedback = ""

for round_no in range(1, 4):

    print("=" * 60)
    print(f"ROUND {round_no}")
    print("=" * 60)

    print("\n🤖 Writer Agent\n")

    code = writer_agent(task, plan, feedback)

    print(code)

    print("\n🤖 Reviewer Agent\n")

    feedback = reviewer_agent(code)

    print(feedback)

    if "APPROVED" in feedback.upper():
        print("\n✅ Reviewer Approved")
        break

ROUND 1

🤖 Writer Agent

```python
# Store student marks
tamil = 90
english = 85
maths = 95
science = 92
social = 88

# Calculate Total
total = tamil + english + maths + science + social

# Calculate Average
total_subjects = 5
average = total / total_subjects

# Calculate Percentage
max_total_marks = 500
percentage = (total / max_total_marks) * 100

# Print the results
print(f"Total: {total}")
print(f"Average: {average}")
print(f"Percentage: {percentage}%")
```

🤖 Reviewer Agent

APPROVED

✅ Reviewer Approved


Executor

In [12]:
executor_agent(code)

EXECUTOR AGENT
Total: 450
Average: 90.0
Percentage: 90.0%


| Agent       | Responsibility             | Input                  | Output               |
| ----------- | -------------------------- | ---------------------- | -------------------- |
| 🤖 Planner  | Breaks the task into steps | User task              | Execution plan       |
| 🤖 Writer   | Generates Python code      | Task + Plan + Feedback | Python code          |
| 🤖 Reviewer | Checks code quality        | Python code            | Approval or feedback |
| 🤖 Executor | Runs the approved code     | Approved Python code   | Program output       |


📋 Planner → Project Manager who creates the implementation plan.
💻 Writer → Software Developer who writes the code.
🔍 Reviewer → Senior Engineer who reviews the code and requests improvements if needed.
⚙️ Executor → Computer that executes the approved program.